# Embeddings: estáticos y contextuales

Un embedding estático da **un** vector por palabra. Lo construimos desde cero (PPMI + SVD) y vemos su límite: «banco» de río y «banco» de dinero comparten vector. Un embedding contextual mezcla la palabra con su contexto (atención) y los separa.

In [1]:
import numpy as np

from common import set_seed, setup_style

setup_style()
rng = set_seed(42)

In [2]:
SENTS = [
    "me senté en el banco del parque junto al río",
    "el banco del río estaba cubierto de hojas",
    "el banco aprobó el crédito y el préstamo",
    "pagué la cuota del crédito en el banco",
    "el río lleva agua al parque",
    "el préstamo tiene una cuota mensual",
] * 5

In [3]:
from itertools import combinations

vocab = sorted({w for s in SENTS for w in s.split()})
idx = {w: i for i, w in enumerate(vocab)}
co = np.zeros((len(vocab), len(vocab)))
for s in SENTS:
    ws = s.split()
    for i, j in combinations(range(len(ws)), 2):
        if j - i <= 3:
            co[idx[ws[i]], idx[ws[j]]] += 1
            co[idx[ws[j]], idx[ws[i]]] += 1
total = co.sum()
pmi = np.log((co * total + 1e-12) / (co.sum(1, keepdims=True) * co.sum(0, keepdims=True) + 1e-12))
ppmi = np.maximum(pmi, 0)
U, S, _ = np.linalg.svd(ppmi)
E = U[:, :8] * S[:8]
E.shape

(26, 8)

In [4]:
def neighbors(w, k=4):
    v = E[idx[w]]
    sims = E @ v / (np.linalg.norm(E, axis=1) * np.linalg.norm(v) + 1e-12)
    return [vocab[i] for i in np.argsort(-sims)[1 : k + 1]]


for w in ["río", "crédito", "banco"]:
    print(w, "→", neighbors(w))

río → ['parque', 'al', 'hojas', 'agua']
crédito → ['y', 'el', 'aprobó', 'banco']
banco → ['me', 'el', 'senté', 'en']


In [5]:
def cos(a, b):
    va, vb = E[idx[a]], E[idx[b]]
    return va @ vb / (np.linalg.norm(va) * np.linalg.norm(vb))


print(f"banco~río {cos('banco', 'río'):.2f} · banco~crédito {cos('banco', 'crédito'):.2f}")

banco~río 0.41 · banco~crédito 0.60


«banco» se parece a la vez a «río» (0,41) y a «crédito» (0,60): un solo vector mezcla los dos sentidos.

## Contextual: atención sobre la oración

Cada palabra pasa a ser un promedio de los vectores de su oración, ponderado por similitud (una cabeza de atención sin entrenar).

In [6]:
from research_paths.attention import scaled_dot_product_attention


def contextual(sentence):
    X = np.stack([E[idx[w]] for w in sentence.split()])
    out, _ = scaled_dot_product_attention(X, X, X)
    return dict(zip(sentence.split(), out, strict=True))


a = contextual("el banco del río estaba cubierto de hojas")["banco"]
b = contextual("el banco aprobó el crédito y el préstamo")["banco"]
cos = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
print(f"coseno entre los dos «banco» contextuales: {cos:.3f} (estático: 1.000)")

coseno entre los dos «banco» contextuales: 0.316 (estático: 1.000)


Un transformer real apila muchas capas de esto con proyecciones aprendidas. El notebook 06 muestra el espacio multilingüe de e5: «aprendizaje por refuerzo» y «reinforcement learning» quedan cerca aunque no compartan palabras.